In [13]:
#NCSG Team 5 Data Dashboard 4/22/26 updated 

from pathlib import Path
import requests
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FRED_API_KEY = "9e5d944924918a26660adf4a492e447e"

OUTPUT_DIR = Path("plotly_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

FORECLOSURE_API_URL = "https://opendata.maryland.gov/resource/w3bc-8mnv.json"

# We search FRED by exact title so you do not need to guess series IDs.
FRED_SERIES_TITLES = {
    "fhfa_hpi": "All-Transactions House Price Index for Maryland",
    "active_listing_count": "Housing Inventory: Active Listing Count in Maryland",
    "median_listing_price": "Housing Inventory: Median Listing Price in Maryland",
    "zhvi": "Zillow Home Value Index (ZHVI) for All Homes Including Single-Family Residences, Condos, and CO-OPs in Maryland",
    "building_permits": "New Private Housing Units Authorized by Building Permits for Maryland"
}

In [14]:
def fred_search_series_id(exact_title: str) -> str:
    url = "https://api.stlouisfed.org/fred/series/search"
    params = {
        "search_text": exact_title,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "limit": 10
    }

    res = requests.get(url, params=params, timeout=30)
    res.raise_for_status()
    payload = res.json()

    series_list = payload.get("seriess", [])
    if not series_list:
        raise ValueError(f"No FRED search results for: {exact_title}")

    # Try exact title match first
    for s in series_list:
        if s.get("title", "").strip().lower() == exact_title.strip().lower():
            return s["id"]

    # Fallback to first result
    return series_list[0]["id"]


def fred_fetch_observations(series_id: str) -> pd.DataFrame:
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json"
    }

    res = requests.get(url, params=params, timeout=30)
    res.raise_for_status()
    payload = res.json()

    obs = payload.get("observations", [])
    if not obs:
        raise ValueError(f"No observations returned for series {series_id}")

    df = pd.DataFrame(obs)
    df = df[df["value"] != "."].copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df[["date", "value"]].dropna().sort_values("date")
    return df

In [15]:
housing_frames = []
resolved_series = {}

for metric_key, exact_title in FRED_SERIES_TITLES.items():
    series_id = fred_search_series_id(exact_title)
    resolved_series[metric_key] = series_id

    df = fred_fetch_observations(series_id)
    df["metric_key"] = metric_key
    df["metric"] = exact_title
    df["series_id"] = series_id
    housing_frames.append(df)

housing_df = pd.concat(housing_frames, ignore_index=True)

print("Resolved FRED series:")
for k, v in resolved_series.items():
    print(f"{k}: {v}")

housing_df.head()

Resolved FRED series:
fhfa_hpi: MDSTHPI
active_listing_count: ACTLISCOUMD
median_listing_price: MEDLISPRIMD
zhvi: MDUCSFRCONDOSMSAMID
building_permits: MDBPPRIVSA


,date,value,metric_key,metric,series_id
0,1975-01-01,61.47,fhfa_hpi,All-Transactions House Price Index for Maryland,MDSTHPI
1,1975-04-01,62.43,fhfa_hpi,All-Transactions House Price Index for Maryland,MDSTHPI
2,1975-07-01,64.88,fhfa_hpi,All-Transactions House Price Index for Maryland,MDSTHPI
3,1975-10-01,66.99,fhfa_hpi,All-Transactions House Price Index for Maryland,MDSTHPI
4,1976-01-01,66.94,fhfa_hpi,All-Transactions House Price Index for Maryland,MDSTHPI


In [16]:
def load_foreclosures(limit=50000):
    params = {
        "$limit": limit,
        "$order": "date ASC"
    }

    res = requests.get(FORECLOSURE_API_URL, params=params, timeout=30)
    res.raise_for_status()
    df = pd.DataFrame(res.json())

    if df.empty:
        raise ValueError("No foreclosure data returned.")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])

    ignore_cols = {"date", "type", ":id", ":created_at", ":updated_at", ":version"}
    county_cols = [c for c in df.columns if c not in ignore_cols]

    for c in county_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    df["statewide_total"] = df[county_cols].sum(axis=1)
    return df[["date", "type", "statewide_total"]].sort_values("date")

foreclosure_df = load_foreclosures()
foreclosure_df.head()

,date,type,statewide_total
0,2021-07-01,Notice of Intent to Foreclose,818.0
1,2021-07-01,Notice of Foreclosure,62.0
2,2021-07-01,Foreclosure Property Registration,65.0
3,2021-08-01,Notice of Intent to Foreclose,1358.0
4,2021-08-01,Notice of Foreclosure,127.0


In [17]:
titles_in_order = [
    FRED_SERIES_TITLES["fhfa_hpi"],
    FRED_SERIES_TITLES["active_listing_count"],
    FRED_SERIES_TITLES["median_listing_price"],
    FRED_SERIES_TITLES["zhvi"],
    FRED_SERIES_TITLES["building_permits"]
]

fig = make_subplots(
    rows=len(titles_in_order),
    cols=1,
    subplot_titles=titles_in_order,
    vertical_spacing=0.05
)

for i, title in enumerate(titles_in_order, start=1):
    df_m = housing_df[housing_df["metric"] == title].sort_values("date")

    if "Listing Price" in title:
        trace = go.Scatter(
            x=df_m["date"],
            y=df_m["value"],
            mode="lines+markers",
            name=title,
            showlegend=False
        )
        y_title = "Dollars"
    elif "Authorized by Building Permits" in title:
        trace = go.Bar(
            x=df_m["date"],
            y=df_m["value"],
            name=title,
            showlegend=False
        )
        y_title = "Units"
    elif "Active Listing Count" in title:
        trace = go.Bar(
            x=df_m["date"],
            y=df_m["value"],
            name=title,
            showlegend=False
        )
        y_title = "Listings"
    elif "Zillow Home Value Index" in title:
        trace = go.Scatter(
            x=df_m["date"],
            y=df_m["value"],
            mode="lines+markers",
            name=title,
            showlegend=False
        )
        y_title = "Index / Dollars"
    else:
        trace = go.Scatter(
            x=df_m["date"],
            y=df_m["value"],
            mode="lines+markers",
            name=title,
            showlegend=False
        )
        y_title = "Index"

    fig.add_trace(trace, row=i, col=1)
    fig.update_yaxes(title_text=y_title, row=i, col=1)

fig.update_layout(
    height=1700,
    title="Maryland Housing Dashboard",
    template="plotly_white"
)

fig.show()

In [7]:
title = FRED_SERIES_TITLES["fhfa_hpi"]
df_m = housing_df[housing_df["metric"] == title].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_m["date"],
    y=df_m["value"],
    mode="lines+markers",
    name="FHFA Home Price Index"
))

fig.update_layout(
    title="Statewide FHFA Home Price Index — Maryland",
    xaxis_title="Date",
    yaxis_title="Index",
    template="plotly_white"
)

fig.show()

In [8]:
title = FRED_SERIES_TITLES["active_listing_count"]
df_m = housing_df[housing_df["metric"] == title].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=df_m["date"],
    y=df_m["value"],
    name="Active Listing Count"
))

fig.update_layout(
    title="Housing Inventory — Active Listing Count (Maryland)",
    xaxis_title="Date",
    yaxis_title="Listings",
    template="plotly_white"
)

fig.show()

In [9]:
title = FRED_SERIES_TITLES["median_listing_price"]
df_m = housing_df[housing_df["metric"] == title].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_m["date"],
    y=df_m["value"],
    mode="lines+markers",
    name="Median Listing Price"
))

fig.update_layout(
    title="Housing Inventory — Median Listing Price (Maryland)",
    xaxis_title="Date",
    yaxis_title="Dollars",
    template="plotly_white"
)

fig.show()

In [10]:
title = FRED_SERIES_TITLES["zhvi"]
df_m = housing_df[housing_df["metric"] == title].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_m["date"],
    y=df_m["value"],
    mode="lines+markers",
    name="ZHVI"
))

fig.update_layout(
    title="Zillow Home Price Index (ZHVI) — Maryland",
    xaxis_title="Date",
    yaxis_title="Dollars / Index",
    template="plotly_white"
)

fig.show()

In [11]:
title = FRED_SERIES_TITLES["building_permits"]
df_m = housing_df[housing_df["metric"] == title].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=df_m["date"],
    y=df_m["value"],
    name="Building Permits"
))

fig.update_layout(
    title="New Private Housing Units Authorized by Building Permits — Maryland",
    xaxis_title="Date",
    yaxis_title="Units",
    template="plotly_white"
)

fig.show()

In [12]:
fig = go.Figure()

preferred_types = [
    "Notice of Intent",
    "Notice of Foreclosure",
    "Foreclosure Property Registration"
]

added_any = False

for t in preferred_types:
    df_t = foreclosure_df[foreclosure_df["type"] == t].sort_values("date")
    if not df_t.empty:
        fig.add_trace(go.Scatter(
            x=df_t["date"],
            y=df_t["statewide_total"],
            mode="lines+markers",
            name=t
        ))
        added_any = True

if not added_any:
    for t in sorted(foreclosure_df["type"].dropna().unique()):
        df_t = foreclosure_df[foreclosure_df["type"] == t].sort_values("date")
        fig.add_trace(go.Scatter(
            x=df_t["date"],
            y=df_t["statewide_total"],
            mode="lines+markers",
            name=t
        ))

fig.update_layout(
    title="Maryland Foreclosures",
    xaxis_title="Date",
    yaxis_title="Statewide Total",
    template="plotly_white"
)

fig.show()